# D2C Fashion Brand — SQL Segmentation & Analysis

In [ ]:
-- ================================================================
--   D2C FASHION BRAND — SQL SEGMENTATION & ANALYSIS
--   Step 2 of 4
-- ================================================================
--
-- WHAT IS SQL DOING HERE?
--   Python gave us a clean table with 24 columns.
--   SQL lets us ask business questions by slicing and grouping
--   that table in structured ways. Think of it like Excel pivot
--   tables, but more precise and repeatable.
--
-- TABLE NAME: customers
-- TOTAL ROWS: 3,900 (one row = one customer)
-- ================================================================


-- ================================================================
--  QUERY 1 — CUSTOMER VALUE TIER PROFILE
--  "What separates High Value customers from Low Value ones?"
-- ================================================================
--
-- WHAT THIS QUERY DOES:
--   Groups all customers into their three tiers (High / Mid / Low)
--   and computes key averages for each group side by side.
--
-- WHY WE'RE DOING THIS:
--   This is the foundation of everything. Before we can build a
--   retention strategy or an ideal customer profile, we need to
--   know: what actually differs between a High Value and Low Value
--   customer? Is it their age? How often they buy? Whether they
--   use discounts? This query answers all of that at once.
--
-- BUSINESS QUESTION ANSWERED:
--   "Who are the genuinely loyal customers vs discount hunters?"
--   "What does the brand's best customer actually look like?"

SELECT
    Customer_Value_Tier,
    COUNT(*)                                                        AS total_customers,
    ROUND(AVG("Purchase Amount (USD)"), 2)                         AS avg_spend,
    ROUND(AVG("Previous Purchases"), 2)                            AS avg_prev_purchases,
    ROUND(AVG(Engagement_Tier), 2)                                 AS avg_engagement_score,
    ROUND(AVG(Promo_Dependency_Score), 2)                          AS avg_promo_dependency,
    SUM(CASE WHEN Discount_Free_Buyer = 1 THEN 1 ELSE 0 END)       AS full_price_buyers,
    ROUND(
        100.0 * SUM(CASE WHEN Discount_Free_Buyer = 1 THEN 1 ELSE 0 END)
        / COUNT(*), 1
    )                                                               AS pct_full_price_buyers,
    ROUND(AVG("Review Rating"), 2)                                  AS avg_satisfaction

FROM customers
GROUP BY Customer_Value_Tier
ORDER BY avg_spend DESC;


-- ================================================================
--  QUERY 2 — LOYAL BUYERS vs DISCOUNT HUNTERS
--  "Is our promo program building loyalty or buying transactions?"
-- ================================================================
--
-- WHAT THIS QUERY DOES:
--   Splits customers by their Promo_Dependency_Score (0 = full
--   price, 2 = used every discount available) and profiles each
--   group's behavior, value, and satisfaction.
--
-- WHY WE'RE DOING THIS:
--   Running discounts costs the brand money. If discount buyers
--   have HIGH previous purchases and HIGH engagement, then
--   discounts are working — they're building real loyalty.
--   If discount buyers have LOW previous purchases and LOW
--   engagement, the brand is just renting their attention one
--   transaction at a time. This query settles that question.
--
-- BUSINESS QUESTION ANSWERED:
--   "Is the discount program actually building a loyal customer
--    base, or just attracting one-time bargain hunters?"

SELECT
    Promo_Dependency_Score,
    CASE Promo_Dependency_Score
        WHEN 0 THEN 'Full-Price Buyer'
        WHEN 1 THEN 'Partial Promo User'
        WHEN 2 THEN 'Full Bargain Hunter'
    END                                                             AS buyer_type,
    COUNT(*)                                                        AS total_customers,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM customers), 1)  AS pct_of_base,
    ROUND(AVG("Purchase Amount (USD)"), 2)                         AS avg_spend,
    ROUND(AVG("Previous Purchases"), 2)                            AS avg_prev_purchases,
    ROUND(AVG(Engagement_Tier), 2)                                 AS avg_engagement,
    ROUND(AVG(Customer_Value_Score), 0)                            AS avg_value_score,
    ROUND(AVG("Review Rating"), 2)                                  AS avg_rating

FROM customers
GROUP BY Promo_Dependency_Score
ORDER BY Promo_Dependency_Score;


-- ================================================================
--  QUERY 3 — CATEGORY vs CUSTOMER TENURE
--  "Which categories attract new customers vs retain veterans?"
-- ================================================================
--
-- WHAT THIS QUERY DOES:
--   Groups by product Category, then shows the average purchase
--   history of customers who buy from each category. Also breaks
--   down by high vs low tenure customers within each category.
--
-- WHY WE'RE DOING THIS:
--   Some categories are "entry points" — new customers discover
--   the brand through them. Others are "retention categories" —
--   customers keep buying from them after they're already loyal.
--   Knowing which is which changes how the brand should market
--   them. Entry-point categories need acquisition spending.
--   Retention categories need loyalty and upsell investment.
--
-- BUSINESS QUESTION ANSWERED:
--   "Which seasons and categories are associated with lower-tenure
--    customers versus those with high previous purchase counts?"

SELECT
    Category,
    COUNT(*)                                                        AS total_customers,
    ROUND(AVG("Previous Purchases"), 2)                            AS avg_prev_purchases,
    ROUND(AVG("Purchase Amount (USD)"), 2)                         AS avg_spend,
    ROUND(AVG(Customer_Value_Score), 0)                            AS avg_value_score,
    ROUND(AVG(Promo_Dependency_Score), 2)                          AS avg_promo_dep,

    -- How many of this category's buyers are "new" (bottom 25% tenure)?
    SUM(CASE WHEN "Previous Purchases" <= 8 THEN 1 ELSE 0 END)     AS low_tenure_buyers,
    ROUND(
        100.0 * SUM(CASE WHEN "Previous Purchases" <= 8 THEN 1 ELSE 0 END)
        / COUNT(*), 1
    )                                                               AS pct_low_tenure,

    -- How many are "veterans" (top 25% tenure)?
    SUM(CASE WHEN "Previous Purchases" >= 40 THEN 1 ELSE 0 END)    AS high_tenure_buyers,
    ROUND(
        100.0 * SUM(CASE WHEN "Previous Purchases" >= 40 THEN 1 ELSE 0 END)
        / COUNT(*), 1
    )                                                               AS pct_high_tenure

FROM customers
GROUP BY Category
ORDER BY avg_prev_purchases DESC;

-- BUSINESS QUESTION ANSWERED:
--   This feeds directly into the Category Funnel panel of your
--   Power BI dashboard — showing which categories are
--   "top of funnel" (new customers) vs "bottom of funnel"
--   (sticky, long-term customers).


-- ================================================================
--  QUERY 4 — SEASON vs PURCHASE BEHAVIOUR
--  "When do our best and worst customers buy?"
-- ================================================================
--
-- WHAT THIS QUERY DOES:
--   Groups purchases by Season and shows the value, promo
--   dependency, and tenure profile of customers in each season.
--
-- WHY WE'RE DOING THIS:
--   If high-value, full-price buyers cluster in Fall and Winter,
--   the brand should concentrate its best (non-discounted) brand
--   campaigns in those seasons. If bargain hunters spike in
--   Summer, that's when promo-dependency is highest — and where
--   the brand needs to be most careful about discount depth.
--

SELECT
    Season,
    COUNT(*)                                                        AS total_customers,
    ROUND(AVG("Purchase Amount (USD)"), 2)                         AS avg_spend,
    ROUND(AVG("Previous Purchases"), 2)                            AS avg_prev_purchases,
    ROUND(AVG(Customer_Value_Score), 0)                            AS avg_value_score,
    ROUND(AVG(Promo_Dependency_Score), 2)                          AS avg_promo_dep,
    SUM(CASE WHEN Discount_Free_Buyer = 1 THEN 1 ELSE 0 END)       AS full_price_buyers,
    ROUND(
        100.0 * SUM(CASE WHEN Discount_Free_Buyer = 1 THEN 1 ELSE 0 END)
        / COUNT(*), 1
    )                                                               AS pct_full_price

FROM customers
GROUP BY Season
ORDER BY avg_value_score DESC;


-- ================================================================
--  QUERY 5 — GEOGRAPHIC OPPORTUNITY MAP
--  "Which states show genuine brand pull vs discount dependency?"
-- ================================================================
--
-- WHAT THIS QUERY DOES:
--   Groups by US state (Location), computes average spend AND
--   promo dependency per state, then labels each state as one
--   of three opportunity types:
--
--   "High Opportunity"  = high spend + low promo dependency
--                         → customers here buy because they love
--                           the brand, NOT because of discounts
--                           → INVEST more marketing here
--
--   "Discount Driven"   = high spend + high promo dependency
--                         → sales exist but are fragile — pull
--                           the discounts and volume may drop
--                           → OPTIMISE — test reducing discounts
--                           gradually
--
--   "Low Priority"      = low spend regardless of promo use
--                         → not commercially interesting yet
--
-- WHY WE'RE DOING THIS:
--   Most brands just look at which states have the most revenue.
--   That's incomplete. A state with $65 avg spend and 35% promo
--   dependency is worth far more than one with $65 avg spend and
--   60% promo dependency — because the first one doesn't need to
--   be bribed.
--
-- HAVING clause: only include states with 30+ customers so
--   averages are statistically meaningful (small samples distort).
--
-- BUSINESS QUESTION ANSWERED:
--   "Which geographies signal organic demand vs discount-driven
--    volume?" and "Which regions are commercially underlevered?"

SELECT
    Location                                                        AS state,
    COUNT(*)                                                        AS total_customers,
    ROUND(AVG("Purchase Amount (USD)"), 2)                         AS avg_spend,
    ROUND(AVG(Promo_Dependency_Score), 2)                          AS avg_promo_dep,
    ROUND(
        100.0 * SUM(CASE WHEN Discount_Free_Buyer = 1 THEN 1 ELSE 0 END)
        / COUNT(*), 1
    )                                                               AS pct_full_price_buyers,
    ROUND(AVG("Previous Purchases"), 2)                            AS avg_tenure,
    ROUND(AVG(Customer_Value_Score), 0)                            AS avg_value_score,

    -- Label each state's opportunity type
    CASE
        WHEN AVG("Purchase Amount (USD)") >= 60
         AND AVG(Promo_Dependency_Score) <= 0.80
        THEN 'High Opportunity'

        WHEN AVG("Purchase Amount (USD)") >= 60
         AND AVG(Promo_Dependency_Score) > 0.80
        THEN 'Discount Driven'

        ELSE 'Low Priority'
    END                                                             AS opportunity_type

FROM customers
GROUP BY Location
HAVING total_customers >= 30
ORDER BY opportunity_type ASC, avg_spend DESC;


-- ================================================================
--  QUERY 6 — SUBSCRIPTION STATUS DEEP DIVE
--  "Do subscribers actually behave better than non-subscribers?"
-- ================================================================
--
-- WHAT THIS QUERY DOES:
--   Compares subscribers vs non-subscribers across every key
--   behavioral metric: spend, tenure, engagement, promo use.
--
-- WHY WE'RE DOING THIS:
--   Subscription programs are expensive to run. If subscribers
--   don't buy more frequently, spend more, or show lower promo
--   dependency than non-subscribers, the subscription program
--   may not be doing what the brand thinks it's doing.
--   This query will tell you if it's worth investing more in
--   subscription acquisition — or if the program needs redesign.
--
-- BUSINESS QUESTION ANSWERED:
--   "What behavioral patterns today predict high customer value
--    over time?" — subscription is one such pattern to test.

SELECT
    "Subscription Status",
    COUNT(*)                                                        AS total_customers,
    ROUND(AVG("Purchase Amount (USD)"), 2)                         AS avg_spend,
    ROUND(AVG("Previous Purchases"), 2)                            AS avg_prev_purchases,
    ROUND(AVG(Engagement_Tier), 2)                                 AS avg_engagement,
    ROUND(AVG(Promo_Dependency_Score), 2)                          AS avg_promo_dep,
    ROUND(
        100.0 * SUM(CASE WHEN Discount_Free_Buyer = 1 THEN 1 ELSE 0 END)
        / COUNT(*), 1
    )                                                               AS pct_full_price,
    ROUND(AVG(Customer_Value_Score), 0)                            AS avg_value_score,
    ROUND(AVG("Review Rating"), 2)                                  AS avg_rating

FROM customers
GROUP BY "Subscription Status"
ORDER BY avg_value_score DESC;


-- ================================================================
--  QUERY 7 — PAYMENT METHOD vs CUSTOMER QUALITY
--  "Does payment method signal anything about customer value?"
-- ================================================================
--
-- WHAT THIS QUERY DOES:
--   Groups by Payment Method and profiles the customer quality
--   behind each one.
--
-- WHY WE'RE DOING THIS:
--   This sounds like a strange question, but payment method is
--   actually a useful proxy for intent and commitment. Credit
--   card buyers often have higher spending limits. PayPal users
--   may be more deal-oriented. Cash buyers (in D2C, this is
--   rare and interesting) could indicate a specific demographic.
--   If one payment method strongly correlates with high value
--   and low promo dependency, the brand can use this in its
--   checkout flow or loyalty program targeting.
--
-- BUSINESS QUESTION ANSWERED:
--   "What does the brand's best customer look like in terms of
--    payment preferences?" (directly from the problem statement)

SELECT
    "Payment Method",
    COUNT(*)                                                        AS total_customers,
    ROUND(AVG("Purchase Amount (USD)"), 2)                         AS avg_spend,
    ROUND(AVG("Previous Purchases"), 2)                            AS avg_tenure,
    ROUND(AVG(Promo_Dependency_Score), 2)                          AS avg_promo_dep,
    ROUND(
        100.0 * SUM(CASE WHEN Discount_Free_Buyer = 1 THEN 1 ELSE 0 END)
        / COUNT(*), 1
    )                                                               AS pct_full_price,
    ROUND(AVG(Customer_Value_Score), 0)                            AS avg_value_score

FROM customers
GROUP BY "Payment Method"
ORDER BY avg_value_score DESC;


-- ================================================================
--  QUERY 8 — IDEAL CUSTOMER PROFILE (ICP)
--  "What does the brand's single best customer type look like?"
-- ================================================================
--
-- WHAT THIS QUERY DOES:
--   Filters to ONLY the brand's top customers — defined as:
--     - High Value tier (top 25% by Customer_Value_Score)
--     - Full-price buyer (no discount, no promo code)
--     - High satisfaction (Review Rating >= 4.0)
--   Then profiles this group in detail across demographics,
--   category, geography, and behaviour.
--
-- WHY WE'RE DOING THIS:
--   This is the output that directly feeds the Retention Playbook.
--   The ICP must be specific enough that a marketing team can
--   use it to make targeting decisions TODAY — on Meta, Google,
--   or email. Vague profiles ("engaged customers who spend well")
--   are useless. This query produces a real description with
--   numbers attached to every characteristic.
--
-- BUSINESS QUESTION ANSWERED:
--   "What does the brand's ideal customer profile look like, and
--    how can it acquire more of them?"

SELECT
    -- Demographics
    ROUND(AVG(Age), 1)                                             AS avg_age,
    MIN(Age)                                                       AS min_age,
    MAX(Age)                                                       AS max_age,

    -- Purchase behaviour
    COUNT(*)                                                       AS icp_customer_count,
    ROUND(AVG("Purchase Amount (USD)"), 2)                        AS avg_spend,
    ROUND(AVG("Previous Purchases"), 2)                           AS avg_prev_purchases,
    ROUND(AVG(Engagement_Tier), 2)                                AS avg_engagement,
    ROUND(AVG(Customer_Value_Score), 0)                           AS avg_value_score,
    ROUND(AVG("Review Rating"), 2)                                AS avg_rating

FROM customers
WHERE Customer_Value_Tier = 'High Value'
  AND Discount_Free_Buyer = 1
  AND "Review Rating" >= 4.0;

-- ── Follow-up: break the ICP down by Gender ──────────────────────
SELECT
    Gender,
    COUNT(*)                                                       AS count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)            AS pct_of_icp,
    ROUND(AVG("Purchase Amount (USD)"), 2)                        AS avg_spend,
    ROUND(AVG("Previous Purchases"), 2)                           AS avg_tenure

FROM customers
WHERE Customer_Value_Tier = 'High Value'
  AND Discount_Free_Buyer = 1
  AND "Review Rating" >= 4.0
GROUP BY Gender
ORDER BY count DESC;

-- ── Follow-up: break the ICP down by Category ────────────────────
SELECT
    Category,
    COUNT(*)                                                       AS count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)            AS pct_of_icp,
    ROUND(AVG("Purchase Amount (USD)"), 2)                        AS avg_spend

FROM customers
WHERE Customer_Value_Tier = 'High Value'
  AND Discount_Free_Buyer = 1
  AND "Review Rating" >= 4.0
GROUP BY Category
ORDER BY count DESC;

-- ── Follow-up: top 10 states in the ICP ──────────────────────────
SELECT
    Location                                                       AS state,
    COUNT(*)                                                       AS icp_customers,
    ROUND(AVG("Purchase Amount (USD)"), 2)                        AS avg_spend

FROM customers
WHERE Customer_Value_Tier = 'High Value'
  AND Discount_Free_Buyer = 1
  AND "Review Rating" >= 4.0
GROUP BY Location
ORDER BY icp_customers DESC
LIMIT 10;

-- ── Follow-up: top payment methods in the ICP ────────────────────
SELECT
    "Payment Method",
    COUNT(*)                                                       AS count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)            AS pct_of_icp

FROM customers
WHERE Customer_Value_Tier = 'High Value'
  AND Discount_Free_Buyer = 1
  AND "Review Rating" >= 4.0
GROUP BY "Payment Method"
ORDER BY count DESC;


-- ================================================================
--  QUERY 9 — PROMOTIONAL SUNSET CANDIDATE SEGMENTS
--  "Which specific segments should we stop discounting first?"
-- ================================================================
--
-- WHAT THIS QUERY DOES:
--   Identifies the exact customer segments that are safest to
--   remove discounts from — because they already demonstrate
--   loyalty signals WITHOUT needing discounts.
--   These are customers who are CURRENTLY receiving discounts
--   but don't need them to stay loyal.
--
-- THE LOGIC:
--   If a customer has:
--     - High Value tier (top 25% by value score)
--     - High engagement (buys Weekly or Fortnightly → tier 5+)
--     - High previous purchases (25+)
--     - High satisfaction (rating ≥ 4.0)
--     - AND is currently using discounts (Promo_Dependency_Score = 2)
--   → They are loyal enough to retain WITHOUT a discount.
--     The brand is spending margin on them unnecessarily.
--
-- WHY THIS MATTERS:
--   You can't just "stop all discounts" — that would cause
--   genuine churn. You need to identify the specific segment
--   where removing discounts carries the lowest risk.
--   These are your sunset candidates.
--
-- BUSINESS QUESTION ANSWERED:
--   "Identify which segments to gradually stop discounting,
--    why those segments specifically, and what the margin
--    impact looks like."

SELECT
    Customer_Value_Tier,
    "Subscription Status",
    COUNT(*)                                                        AS sunset_candidates,
    ROUND(AVG("Purchase Amount (USD)"), 2)                        AS avg_spend,
    ROUND(AVG("Previous Purchases"), 2)                           AS avg_prev_purchases,
    ROUND(AVG(Engagement_Tier), 2)                                AS avg_engagement,
    ROUND(AVG("Review Rating"), 2)                                AS avg_rating,

    -- Estimated margin recovered per customer if discount (~20% of spend) removed
    ROUND(AVG("Purchase Amount (USD)") * 0.20, 2)                 AS est_margin_per_customer,

    -- Total estimated margin recovery for this segment
    ROUND(COUNT(*) * AVG("Purchase Amount (USD)") * 0.20, 0)      AS total_margin_recovery_est

FROM customers
WHERE Promo_Dependency_Score = 2          -- Currently using full discounts
  AND Customer_Value_Tier = 'High Value'  -- Top 25% by value
  AND "Previous Purchases" >= 25          -- Long-term customers
  AND "Review Rating" >= 4.0             -- Satisfied customers
  AND Engagement_Tier >= 4               -- Buy at least bi-weekly
GROUP BY Customer_Value_Tier, "Subscription Status"
ORDER BY total_margin_recovery_est DESC;

-- NOTE ON MARGIN ESTIMATE:
--   The 20% discount assumption is a placeholder. Replace with
--   your actual average discount depth once known. The formula
--   is: avg_spend × discount_rate × number_of_customers.


-- ================================================================
--  QUERY 10 — SUMMARY SCORECARD
--  One clean table the founding team can review in 60 seconds
-- ================================================================
--
-- WHAT THIS QUERY DOES:
--   Pulls the single most important number from each analysis
--   into one summary view. This is what you present first in
--   any founder meeting — before showing detailed breakdowns.
--
-- WHY WE'RE DOING THIS:
--   Founders need to orient quickly. Showing 9 queries at once
--   loses them. This scorecard gives them the 10 headline numbers
--   and they can drill into whichever ones surprise them.

SELECT '1. Total Customers'              AS metric, COUNT(*) AS value FROM customers
UNION ALL
SELECT '2. High Value Customers',         COUNT(*) FROM customers WHERE Customer_Value_Tier = 'High Value'
UNION ALL
SELECT '3. Full-Price Buyers',            COUNT(*) FROM customers WHERE Discount_Free_Buyer = 1
UNION ALL
SELECT '4. Full Bargain Hunters',         COUNT(*) FROM customers WHERE Promo_Dependency_Score = 2
UNION ALL
SELECT '5. Pct Bargain Hunters (%)',      ROUND(100.0*COUNT(*)/3900, 1) FROM customers WHERE Promo_Dependency_Score = 2
UNION ALL
SELECT '6. Avg Spend All Customers ($)',  ROUND(AVG("Purchase Amount (USD)"), 2) FROM customers
UNION ALL
SELECT '7. Avg Spend High Value ($)',     ROUND(AVG("Purchase Amount (USD)"), 2) FROM customers WHERE Customer_Value_Tier = 'High Value'
UNION ALL
SELECT '8. Avg Previous Purchases',       ROUND(AVG("Previous Purchases"), 2) FROM customers
UNION ALL
SELECT '9. High Satisfaction Customers',  COUNT(*) FROM customers WHERE Satisfaction_Flag = 'High'
UNION ALL
SELECT '10. ICP Customers (ideal)',       COUNT(*) FROM customers
    WHERE Customer_Value_Tier = 'High Value'
      AND Discount_Free_Buyer = 1
      AND "Review Rating" >= 4.0;

